# Блок 2. Практика: Vector и парсинг логов

В этом notebook мы:
1. Запустим генератор логов и Vector
2. Научимся парсить различные форматы логов (JSON, Combined, BIND, CEF)
3. Потренируемся с VRL для трансформаций
4. Настроим отправку данных в PostgreSQL

## Часть 1: Подготовка окружения

Перед началом работы убедитесь, что:
1. PostgreSQL запущен (из Блока 1)
2. Генератор логов настроен и готов к запуску
3. Vector установлен и готов к работе

In [ ]:
# Проверим версию Vector
!docker run --rm timberio/vector:0.52.0-alpine --version

# Проверим, что PostgreSQL запущен
!docker ps | grep postgres || echo "PostgreSQL не запущен. Запустите его из Блока 1."

## Часть 2: Форматы логов в FinanceFlow

В инфраструктуре FinanceFlow используются четыре типа логов:

1. **Auth Events** — JSON формат
2. **Nginx Logs** — Combined Log Format
3. **DNS Queries** — BIND query log
4. **Firewall Events** — CEF формат

Давайте посмотрим на примеры каждого формата:

In [ ]:
# Примеры форматов логов

# 1. Auth Events (JSON)
auth_example = '{"timestamp": "2024-03-08T03:47:22.456Z", "event_type": "login_success", "username": "dev_sergey", "source_ip": "192.168.1.100", "success": true, "details": {"method": "password"}}'

# 2. Nginx Logs (Combined Log Format)
nginx_example = '203.0.113.42 - - [01/Mar/2024:10:15:32 +0000] "GET /admin HTTP/1.1" 403 162 "-" "Mozilla/5.0"'

# 3. DNS Queries (BIND query log)
dns_example = '12-Mar-2024 14:23:45.123 client @0x7f9b2c001a00 192.168.1.100#54321 (x7kj2m9p.data-sync.xyz): query: x7kj2m9p.data-sync.xyz IN A +E(0)K (8.8.8.8)'

# 4. Firewall Events (CEF)
firewall_example = 'Mar 01 10:15:32 fw-gateway CEF:0|FinanceFlow|Firewall|1.0|200|Connection block|7|src=203.0.113.42 dst=192.168.1.10 spt=45678 dpt=22 proto=TCP act=block reason=port_scan'

print("Auth Events (JSON):")
print(auth_example)
print("\nNginx Logs (Combined):")
print(nginx_example)
print("\nDNS Queries (BIND):")
print(dns_example)
print("\nFirewall Events (CEF):")
print(firewall_example)

## Часть 3: Парсинг JSON (Auth Events)

Начнём с самого простого формата — JSON. Auth Events уже в JSON, но нужно распарсить их из поля `.message`.

### TODO: Задание 3.1 — Парсинг JSON

Создайте конфигурацию для парсинга auth_events:

1. Используйте `parse_json!` для парсинга JSON из `.message`
2. Преобразуйте timestamp в стандартный формат
3. Добавьте поле `severity` на основе `event_type`

Попробуйте написать VRL-код для парсинга:

In [ ]:
# Тестируем парсинг JSON
import subprocess

auth_event = '{"timestamp": "2024-03-08T03:47:22.456Z", "event_type": "login_success", "username": "dev_sergey", "source_ip": "192.168.1.100", "success": true}'

# TODO: Напишите VRL-код для парсинга JSON
# Шаг 1: Распарсите JSON из .message используя parse_json!(.message)
# Шаг 2: Преобразуйте timestamp используя parse_timestamp!(.timestamp, format: "%+")
# Шаг 3: Добавьте severity на основе event_type:
#   - если event_type == "login_failure" -> severity = "warning"
#   - иначе -> severity = "info"
vrl_code = '''
# Ваш код здесь
# . = parse_json!(.message)
# .timestamp = parse_timestamp!(.timestamp, format: "%+")
# .severity = if .event_type == "login_failure" { "warning" } else { "info" }
'''

result = subprocess.run(
    ['docker', 'run', '--rm', '-i', 'timberio/vector:0.52.0-alpine', 'vrl', vrl_code],
    input=f'{{"message": "{auth_event}"}}',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("Ошибка:", result.stderr)

## Часть 4: Парсинг Combined Log Format (Nginx Logs)

Nginx использует Combined Log Format. Нужно распарсить его с помощью regex.

### TODO: Задание 4.1 — Парсинг Combined Log Format

Формат: `IP - - [timestamp] "method path protocol" status size "referer" "user_agent"`

Напишите regex для извлечения всех полей:

In [ ]:
# Тестируем парсинг nginx-логов
import subprocess

nginx_log = '203.0.113.42 - - [01/Mar/2024:10:15:32 +0000] "GET /admin HTTP/1.1" 403 162 "-" "Mozilla/5.0"'

# TODO: Напишите regex для парсинга всех полей
# Формат: IP - - [timestamp] "method path protocol" status size "referer" "user_agent"
# Подсказка: используйте именованные группы (?P<name>pattern)
# Нужно извлечь: source_ip, timestamp, method, path, status, size, referer, user_agent
regex_pattern = r''  # Ваш regex здесь

vrl_code = f'''
# TODO: Распарсите с помощью parse_regex!
# TODO: Преобразуйте status и size в числа используя to_int!
# TODO: Парсите timestamp используя format: "%d/%b/%Y:%H:%M:%S %z"
# TODO: Добавьте severity на основе status (>=400 -> "error", иначе "info")
# . |= parse_regex!(.message, r"{regex_pattern}")
# .status = to_int!(.status)
# .size = to_int!(.size)
# .timestamp = parse_timestamp!(.timestamp, format: "%d/%b/%Y:%H:%M:%S %z")
# .severity = if .status >= 400 {{ "error" }} else {{ "info" }}
'''

result = subprocess.run(
    ['docker', 'run', '--rm', '-i', 'timberio/vector:0.52.0-alpine', 'vrl', vrl_code],
    input=f'{{"message": "{nginx_log}"}}',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("Ошибка:", result.stderr)

## Часть 5: Парсинг BIND Query Log (DNS Queries)

DNS-запросы в формате BIND query log требуют более сложного парсинга.

### TODO: Задание 5.1 — Парсинг BIND Query Log

Формат: `timestamp client @client_id source_ip#port (domain): query: domain IN TYPE flags (resolver)`

Напишите regex для извлечения полей:

In [ ]:
# Тестируем парсинг DNS-запросов
import subprocess

dns_log = '12-Mar-2024 14:23:45.123 client @0x7f9b2c001a00 192.168.1.100#54321 (x7kj2m9p.data-sync.xyz): query: x7kj2m9p.data-sync.xyz IN A +E(0)K (8.8.8.8)'

# TODO: Напишите regex для парсинга BIND формата
# Формат: timestamp client @0x... source_ip#port (domain): query: domain IN TYPE +flags (resolver)
# Нужно извлечь: timestamp, source_ip, source_port, query_domain, query_type, resolver (опционально)
regex_pattern = r''  # Ваш regex здесь

vrl_code = f'''
# TODO: Распарсите с помощью parse_regex!
# TODO: Преобразуйте source_port в число
# TODO: Парсите timestamp используя format: "%d-%b-%Y %H:%M:%S%.f"
# TODO: Добавьте severity = "info"
# . |= parse_regex!(.message, r"{regex_pattern}")
# .source_port = to_int!(.source_port)
# .timestamp = parse_timestamp!(.timestamp, format: "%d-%b-%Y %H:%M:%S%.f")
# .severity = "info"
'''

result = subprocess.run(
    ['docker', 'run', '--rm', '-i', 'timberio/vector:0.52.0-alpine', 'vrl', vrl_code],
    input=f'{{"message": "{dns_log}"}}',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("Ошибка:", result.stderr)

## Часть 6: Парсинг CEF (Firewall Events)

CEF (Common Event Format) — самый сложный формат. Нужно парсить и основные поля, и extensions (key=value пары).

### TODO: Задание 6.1 — Парсинг CEF

Формат: `timestamp hostname CEF:version|vendor|product|version|event_id|action|severity|extensions`

Нужно:
1. Извлечь основные поля (timestamp, severity)
2. Парсить extensions с помощью `parse_key_value!`
3. Извлечь поля из extensions (src, dst, spt, dpt, proto, act, reason)

In [ ]:
# Тестируем парсинг CEF
import subprocess

firewall_log = 'Mar 01 10:15:32 fw-gateway CEF:0|FinanceFlow|Firewall|1.0|200|Connection block|7|src=203.0.113.42 dst=192.168.1.10 spt=45678 dpt=22 proto=TCP act=block reason=port_scan'

# TODO: Напишите VRL-код для парсинга CEF
# Формат: timestamp hostname CEF:version|vendor|product|version|event_id|action|severity|extensions
# Шаг 1: Извлеките timestamp, severity и extensions с помощью parse_regex!
# Шаг 2: Парсите timestamp используя format: "%b %d %H:%M:%S"
# Шаг 3: Парсите extensions используя parse_key_value! с разделителями " " и "="
# Шаг 4: Извлеките поля из extensions: src, dst, spt, dpt, proto, act, reason
# Шаг 5: Преобразуйте severity из числа в строку (>=7 -> "error", >=4 -> "warning", иначе "info")
vrl_code = '''
# Ваш код здесь
# . |= parse_regex!(.message, r"^...")
# .timestamp = parse_timestamp!(.timestamp, format: "%b %d %H:%M:%S")
# .extensions = parse_key_value!(.extensions, field_delimiter: " ", key_value_delimiter: "=")
# .source_ip = .extensions.src
# ...
'''

result = subprocess.run(
    ['docker', 'run', '--rm', '-i', 'timberio/vector:0.52.0-alpine', 'vrl', vrl_code],
    input=f'{{"message": "{firewall_log}"}}',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("Ошибка:", result.stderr)

## Часть 7: Проверка конфигурации Vector

Перед запуском всегда проверяйте конфигурацию:

In [ ]:
# Проверяем конфигурацию Vector
# TODO: Создайте конфигурацию vector.toml и проверьте её

# Пример проверки:
# !docker run --rm -v $(pwd)/vector-configs:/config timberio/vector:0.52.0-alpine vector validate /config/vector.toml

## Часть 8: Шпаргалка по VRL

**Парсинг:**
- `parse_json!(.field)` — парсинг JSON
- `parse_regex!(.field, r"pattern")` — regex с именованными группами
- `parse_key_value!(.field, field_delimiter: " ", key_value_delimiter: "=")` — формат key=value

**Преобразование типов:**
- `to_int!(.field)` — в число
- `to_string(.field)` — в строку

**Время:**
- `parse_timestamp!(.field, format: "%d/%b/%Y:%H:%M:%S %z")` — парсинг даты
- Форматы: `%+` (ISO 8601), `%d/%b/%Y:%H:%M:%S %z` (nginx), `%d-%b-%Y %H:%M:%S%.f` (BIND), `%b %d %H:%M:%S` (CEF)

**Условия:**
- `if condition { ... } else { ... }` — условие
- `exists(.field)` — поле существует

**Удаление полей:**
- `del(.field)` — удалить поле

## Часть 9: Проверка данных в PostgreSQL

После настройки Vector проверьте, что данные записываются в PostgreSQL:

In [ ]:
# TODO: Проверьте наличие данных в PostgreSQL
# Подключитесь к базе и выполните запросы:

# SELECT COUNT(*) FROM auth_events;
# SELECT COUNT(*) FROM nginx_logs;
# SELECT COUNT(*) FROM dns_queries;
# SELECT COUNT(*) FROM firewall_events;

# Пример подключения через psql:
# !docker exec -it security-postgres psql -U analyst -d security_logs -c "SELECT COUNT(*) FROM auth_events;"

## Что дальше?

После успешной настройки Vector вы сможете:
- Централизованно собирать логи из разных источников
- Парсить различные форматы логов
- Отправлять данные в PostgreSQL для последующего анализа

В следующем блоке мы научимся оптимизировать хранение данных и создавать индексы для быстрого поиска индикаторов атаки.

## Полезные команды

```bash
# Проверить конфиг Vector
docker run --rm -v $(pwd)/vector-configs:/config timberio/vector:0.52.0-alpine vector validate /config/vector.toml

# Запустить Vector
docker run --rm -v $(pwd)/vector-configs:/config timberio/vector:0.52.0-alpine vector --config /config/vector.toml

# Тестировать VRL с данными
echo '{"message": "..."}' | docker run --rm -i timberio/vector:0.52.0-alpine vrl 'ваш_код'

# Проверить данные в PostgreSQL
docker exec -it security-postgres psql -U analyst -d security_logs -c "SELECT COUNT(*) FROM auth_events;"
```